# Исследовательский анализ датасета KITTI
Ноутбук выполняет разведочный анализ (EDA): распределение классов, размеры изображений, размеры рамок и визуализацию примеров с разметкой.
Предполагается, что данные уже сконвертированы: `python main.py --prepare-data`.

In [ ]:
import json, os
from collections import Counter
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

ANN = '../data/processed/coco/instances_train.json'
IMAGES = '../data/raw/kitti/training/image_2'
coco = COCO(ANN)
cats = {c['id']: c['name'] for c in coco.loadCats(coco.getCatIds())}
print('Классы:', cats)
print('Изображений:', len(coco.imgs), '| Объектов:', len(coco.anns))

## Распределение объектов по классам

In [ ]:
counts = Counter(a['category_id'] for a in coco.anns.values())
names = [cats[k] for k in sorted(counts)]
values = [counts[k] for k in sorted(counts)]
plt.bar(names, values, color='#4C72B0')
plt.title('Распределение объектов по классам (train)')
plt.ylabel('Количество объектов')
plt.show()

## Распределение площадей рамок (анализ масштабов объектов)

In [ ]:
import numpy as np
areas = np.array([a['area'] for a in coco.anns.values()])
plt.hist(np.sqrt(areas), bins=50, color='#55A868')
plt.xlabel('sqrt(площадь рамки), px')
plt.ylabel('Частота')
plt.title('Распределение размеров объектов')
plt.show()
print('Доля малых объектов (sqrt(area) < 32 px): %.1f%%' % (100 * (np.sqrt(areas) < 32).mean()))

## Визуализация примеров с разметкой

In [ ]:
from PIL import Image
img_ids = list(coco.imgs.keys())[:3]
for img_id in img_ids:
    info = coco.loadImgs(img_id)[0]
    img = Image.open(os.path.join(IMAGES, info['file_name']))
    plt.figure(figsize=(12, 4)); plt.imshow(img); ax = plt.gca()
    for ann in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color='red', linewidth=2))
        ax.text(x, y - 4, cats[ann['category_id']], color='white', fontsize=8, bbox=dict(facecolor='red', alpha=0.7))
    plt.axis('off'); plt.show()